# Homework 1

## Importing necessary libraries

In [ ]:
from PIL import Image
from glob import glob
import time
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, Dataset, DataLoader
from torchsummary import summary
from torchvision import transforms as T
from tensorflow import summary as tfsummary
import pickle
from sklearn.metrics import classification_report
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import clear_output
import matplotlib.pyplot as plt
import os

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Reading training and test samples

Instructions for downloading and uploading photos to Colab can be found on GitHub

In [ ]:
# Image size
heigth_width = 32

CLASSES = ['Торт', 'Ласточка', 'Кошка'] # You need to specify your classes here

# Images - training and test
images = []
images_t = []
# Classes - training and test
classes = []
classes_t = []

#
# YOUR DATA READING CODE
#

train_X = np.array(images)
train_y = np.array(classes)

test_X = np.array(images_t)
test_y = np.array(classes_t)

### Displaying an example image

In [ ]:
Image.fromarray(train_X[99]).resize((512,512))

## Creating Transform, Dataset, and PyTorch DataLoader

In [ ]:
# TRANSFORM

transform = T.Compose([
    # ...
])

# # Displaying the transformed image
# Image.fromarray((transform(torch.Tensor(train_X[50]).permute(2, 0, 1)/255.).\
#                  permute(1, 2, 0).numpy()*255.).astype(np.uint8)).\
#                  resize((256, 256))

In [ ]:
#
# Your Dataset-class applying transform
#

class HW1_Dataset(Dataset):
  def __init__(self, X, y, transform=None, p=0.0):
    assert X.size(0) == y.size(0)

In [ ]:
batch_size = 32
dataloader = {}

for (X, y), part in zip([(train_X, train_y), (test_X, test_y)],
                        ['train', 'test']):
    tensor_x = torch.Tensor(X)
    tensor_y = F.one_hot(torch.Tensor(y).to(torch.int64),
                         num_classes=len(CLASSES))/1.

    # creating a dataset object
    dataset = HW1_Dataset(tensor_x, tensor_y,
                          transform if part=='train' else None,
                          p=0.5)

    # creating an instance of the DataLoader class
    dataloader[part] = DataLoader(dataset, batch_size=batch_size,
                                  shuffle=True
                                  # ... additional parameters if needed ...
                                  )

dataloader

## Creating the model

In [ ]:
class Normalize(nn.Module):
    def __init__(self, mean, std):
        super(Normalize, self).__init__()
        self.mean = torch.tensor(mean).to(device)
        self.std = torch.tensor(std).to(device)

    def forward(self, input):
        x = input / 255.0
        x = x - self.mean
        x = x / self.std
        return x.permute(0, 3, 1, 2) # nhwc -> nm

### Choose: custom or fine-tuning

In [ ]:
# CUSTOM MODEL
HIDDEN_SIZE = 48

class HW1_MLP(nn.Module):
    def __init__(self, hidden_size=32, classes=100, mean=None, std=None):
        super(HW1_MLP, self).__init__()

        self.seq = nn.Sequential(
            # ...
        )

    def forward(self, input):
        x = self.norm(input)
        return self.seq(x)

model = HW1_MLP(
    hidden_size=HIDDEN_SIZE,
    classes=len(CLASSES),
    # mean=custom_mean,
    # std=custom_std
)
model.to(device)


# PRE-TRAINED MODEL
model_loaded = torch.hub.load("chenyaofo/pytorch-cifar-models",
                              "cifar100_mobilenetv2_x0_5",
                              #'cifar100_resnet20',
                              pretrained=True)

model = nn.Sequential(
    Normalize([0.5356, 0.5012, 0.4595], [0.2022, 0.2025, 0.2077]),
    model_loaded
).to(device)

In [ ]:
# FOR PRETRAINED: Freezing weights

## Choosing the loss function and gradient descent optimizer

In [ ]:
# Your loss function, optimizer, and setup
criterion = None
optimizer = None

## Model training by epochs

To speed up training, move to `device`

In [ ]:
EPOCHS = 250

steps_per_epoch = len(dataloader['train'])
steps_per_epoch_val = len(dataloader['test'])

for epoch in range(EPOCHS): # iterate over the dataset multiple times
    running_loss = 0.0
    model.train()
    for i, batch in enumerate(dataloader['train'], 0):
        # get one mini-batch; batch is a two-element list of [inputs, labels]
        inputs, labels = batch

        # clear previous gradients from the last iteration
        optimizer.zero_grad()

        # forward + backward passes + optimization
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # for statistical calculations
        running_loss += loss.item()

    print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / steps_per_epoch:.3f}')
    running_loss = 0.0
    model.eval()
    with torch.no_grad(): # disable automatic differentiation
        for i, data in enumerate(dataloader['test'], 0):
            inputs, labels = data

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
    print(f'[{epoch + 1}, {i + 1:5d}] val loss: {running_loss / steps_per_epoch_val:.3f}')

print('Training finished')

## Checking model quality by classes on training and test samples

In [ ]:
for part in ['train', 'test']:
    y_pred = []
    y_true = []
    with torch.no_grad(): # disable automatic differentiation
        for i, data in enumerate(dataloader[part], 0):
            inputs, labels = data

            outputs = model(inputs).detach().numpy()
            y_pred.append(outputs)
            y_true.append(labels.numpy())
        y_true = np.concatenate(y_true)
        y_pred = np.concatenate(y_pred)
        print(part)
        print(classification_report(y_true.argmax(axis=-1), y_pred.argmax(axis=-1),
                                    digits=4, target_names=list(map(str, CLASSES))))
        print('-'*55)

## Saving the model to ONNX

In [ ]:
!pip install onnx onnxscript onnxruntime

In [ ]:
# Input tensor for the model
x = torch.randn(1, 3, heigth_width, heigth_width, requires_grad=True).to(device)
torch_out = model(x)

# Export the model
torch.onnx.export(model,               # model
                  x,                   # input tensor (or tuple of multiple tensors)
                  "cifar100_CNN.onnx", # where to save (either file path or fileObject)
                  export_params=True,  # saves the weights of trained parameters inside the model file
                  opset_version=18,    # ONNX version
                  dynamo=False,        # disables the new torch.onnx.dynamo exporter
                  do_constant_folding=True,  # whether to perform constant folding for optimization
                  input_names = ['input'],   # input layer name
                  output_names = ['output'],  # output layer name
                  dynamic_axes={'input' : {0 : 'batch_size'},    # dynamic axes, in this case only batch size
                                'output' : {0 : 'batch_size'}})